<a href="https://colab.research.google.com/github/kader-xai/llm-from-scratch/blob/main/notebooks/answer/Chapter09_transformer_block_answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧱 **LLM From Scratch · Chapter 09 — Transformer Block — attention + FFN + residuals**

> ✅ Answer notebook — full working code.

## Goal

A **Transformer block** combines everything with the **Pre-LN residual** pattern used by GPT-2+:

```
x = x + MultiHeadAttention(LayerNorm(x))
x = x + FeedForward(LayerNorm(x))
```

The `x +` (residual / skip connection) lets gradients flow through deep stacks; the LayerNorm sits
*inside* the residual (Pre-LN) for stable training. Stacking N of these blocks is the model's depth.

In [ ]:
# ── Setup (run first) ─────────────────────────────────────────────────
# Pure PyTorch — no extra installs needed beyond torch (preinstalled on Colab).
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

In [ ]:
# Re-define the pieces from Ch 05-08 (so this notebook is self-contained)
class Head(nn.Module):
    def __init__(s, C, hs, bs):
        super().__init__()
        s.key=nn.Linear(C,hs,bias=False); s.query=nn.Linear(C,hs,bias=False); s.value=nn.Linear(C,hs,bias=False)
        s.register_buffer("tril", torch.tril(torch.ones(bs,bs))); s.scale=hs**-0.5
    def forward(s,x):
        B,T,C=x.shape; k,q,v=s.key(x),s.query(x),s.value(x)
        a=(q@k.transpose(-2,-1)*s.scale).masked_fill(s.tril[:T,:T]==0,float("-inf"))
        return F.softmax(a,-1)@v
class MultiHeadAttention(nn.Module):
    def __init__(s,C,nh,bs):
        super().__init__(); s.heads=nn.ModuleList([Head(C,C//nh,bs) for _ in range(nh)]); s.proj=nn.Linear(C,C)
    def forward(s,x): return s.proj(torch.cat([h(x) for h in s.heads],-1))
class FeedForward(nn.Module):
    def __init__(s,C): super().__init__(); s.net=nn.Sequential(nn.Linear(C,4*C),nn.GELU(),nn.Linear(4*C,C))
    def forward(s,x): return s.net(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, C, n_head, block_size):
        super().__init__()
        self.sa  = MultiHeadAttention(C, n_head, block_size)
        self.ff  = FeedForward(C)
        self.ln1 = nn.LayerNorm(C)
        self.ln2 = nn.LayerNorm(C)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))      # Pre-LN residual around attention
        x = x + self.ff(self.ln2(x))      # Pre-LN residual around FFN
        return x

block = Block(C=32, n_head=4, block_size=8)
print("Block output:", block(torch.randn(4, 8, 32)).shape)   # (4, 8, 32) — shape preserved

✅ The repeating unit is done. Next: turn the final hidden state into vocab logits.